/***************************************************************************************************
Procedure:          MDD Patient
Create Date:        2025-06-26
Author:             Naveen Deevi
Description:        This table is for Bio/nomics data table creating with the help of Felice, Elene & Erwin. This specific table is for Erwin's individual ask to pull the patient level data. This table has one row for one patient.
Customer:           Felice Sheih
Affected by table(s): Patients, Patient_Providers, providers, Provider_practices, Practice, Onboarding_metadata, patient_diagnosis
                    
Used By:            Data Science
Parameter(s):       @Patient ID created date - When the patient has onboarded to the Headlamp system. 
                    @Practice Name - Practice name of the user. 
                    @Provider name - Provider name of the user. 
Inclusion Critera: 
    1. Only Patients who have MDD as the Dx.  

Exclusion Critera: 
    1. 
    
Assumptions: 
    1. Need to be confirmed. If stopped date for a condition is not available then as per common notion 12 months are added to the started date. 
    2. 
    
****************************************************************************************************
SUMMARY OF CHANGES
Date(yyyy-mm-dd)    Author              Comments
------------------- ------------------- ------------------------------------------------------------
2025-Jun-26.        Naveen.             On Jun 25, Erwin asked for removal or omitting of Non-depressive drugs. So removing them. 
***************************************************************************************************/

In [1]:
import pandas as pd
import spacy as spc
import scispacy as scspcy
# Multiple ways to get and work with home directory
import os
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
import requests
import re
from collections import Counter

# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Datasets"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")
#files = os.listdir(os.getcwd())
#print(files)

# Step 3: Download and load the model
try:
    nlp = spc.load("en_core_sci_sm")
    
    # Test the model
    text = "Paracetamol is commonly used to treat fever and mild to moderate pain."
    doc = nlp(text)
    
    # Print results
    print("Named Entities:")
    for ent in doc.ents:
        print(f"Entity: {ent.text}, Label: {ent.label_}")
        
except Exception as e:
    print(f"Error: {e}")
    print("\nTroubleshooting steps:")
    print("1. Make sure you're running this in a fresh Python session")
    print("2. Try restarting your Jupyter kernel")
    print("3. Verify your Python environment has write permissions")




Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi

Contents of home directory:
Current working directory: /Users/naveendeevi/Datasets
Named Entities:
Entity: Paracetamol, Label: ENTITY
Entity: treat, Label: ENTITY
Entity: fever, Label: ENTITY
Entity: mild, Label: ENTITY
Entity: moderate pain, Label: ENTITY


/opt/anaconda3/envs/NLP/lib/python3.12/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [2]:
## Functions

nlp = spc.load("en_ner_bc5cdr_md")

def text_to_disease(drug_name):
    #print(f"Processing: {drug_name}")
    
    # Construct FDA URL
    fda_url = f'https://api.fda.gov/drug/label.json?search=openfda.brand_name:"{drug_name.lower()}"'
    
    try:
        fda_resp = requests.get(fda_url).json()
        results = fda_resp.get('results', [])

        if results and 'indications_and_usage' in results[0]:
            indication_large_body = results[0]['indications_and_usage'][0]
        else:
            return "No indications info found", None
    except Exception as e:
        print(f"Error fetching FDA data for {drug_name}: {e}")
        return "No indications info found", None

    # Clean and process text
    text_clean = ''.join(c.lower() if c.isalpha() else c for c in indication_large_body)
    result = nlp(text_clean)
    result_ents = [str(ent) for ent in result.ents if str(ent).lower() != drug_name.lower()]

    # Count and sort
    counted = Counter(result_ents)
    sorted_result = sorted(counted.items(), key=lambda x: x[1], reverse=True)

    return indication_large_body, sorted_result

# Safe wrapper to ensure consistency
def safe_text_to_disease(drug_name):
    #print(drug_name)
    try:
        output = text_to_disease(drug_name)
        if isinstance(output, tuple) and len(output) == 2:
            return output
        else:
            return "No indications info found", None
    except Exception as e:
        print(f"Exception in processing {drug_name}: {e}")
        return "No indications info found", None



In [213]:
df = pd.read_csv('MDD_patient_per_rows_V4.csv')

In [214]:
df.shape

(3333, 15)

In [52]:
df.head(4)

,patient_id,preferred_name,gender,date_of_birth,pat_age,pat_created_date,practice_name,provider_name,onboarding_status,rxcui,medication_name,dose,ingredient_name,indication_terms_fda,number_of_refills
0,MTIzNDU2Nzg5MGFiY2RlZg0DoV1+PX+XRLRxIy1FcBS02k...,Joseph Stanley,PATIENT_GENDER_FEMALE,1956-01-14,69,"March 7, 2025, 4:54 PM",Tjalas Healthcare Consulting LLC,Kirk Tjalas,PASSWORD_CREATED,235988.0,venlafaxine hydrochloride,NaN,venlafaxine hydrochloride,"[('anxiety disorder', 3), ('depressive disorde...",1
1,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8On...,Diana Shelton,PATIENT_GENDER_MALE,1994-03-07,31,"January 3, 2025, 1:29 PM",Matthew A Berger MD PC,Matthew Berger,ONBOARDED,1086768.0,vilazodone hydrochloride,10 mg,vilazodone hydrochloride,"[('depressive disorder', 2), ('mdd', 2)]",1
2,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8On...,Diana Shelton,PATIENT_GENDER_MALE,1994-03-07,31,"January 3, 2025, 1:29 PM",Matthew A Berger MD PC,Matthew Berger,ONBOARDED,1099870.0,divalproex sodium 500 mg delayed release oral ...,NaN,divalproex sodium,"[('divalproex sodium extended-release', 5), ('...",9
3,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8On...,Diana Shelton,PATIENT_GENDER_MALE,1994-03-07,31,"January 3, 2025, 1:29 PM",Matthew A Berger MD PC,Matthew Berger,ONBOARDED,135423.0,zyprexa,NaN,olanzapine pamoate,"[('schizophrenia', 4), ('schizophrenia zyprexa...",20


In [53]:
#df.shape
# 3333,15 --> 3,333 rows
(df[df['indication_terms_fda'].isna()]).shape # 1107,15
# of which indication_terms_fda is null for 1107, which is 20%

(654, 15)

In [54]:
mask = df['indication_terms_fda'].isna()

# Apply the function and extract the first result to fill the missing values
df.loc[mask, 'indication_terms_fda'] = df.loc[mask, 'ingredient_name'].apply(
    lambda x: safe_text_to_disease(x)[1]
)

Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'float' object has no attribute 'lower'
Exception in processing nan: 'floa

In [55]:
df[df['indication_terms_fda'].isna()].shape

(534, 15)

In [112]:
df.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'rxcui', 'medication_name', 'dose',
       'ingredient_name', 'indication_terms_fda', 'number_of_refills',
       'is_depression_related'],
      dtype='object')

In [56]:
# Extracting the first word from the name of the medication name, as it seems to be most nearest to the pull indications
med_text = set(df.loc[df['indication_terms_fda'].isna(),'medication_name'])
med_text_2 = [] 
for i in med_text:
    first_word = i.strip().split()[0]
    med_text_2.append(first_word)
med_text_2 = set(med_text_2)

In [57]:
med_text_dict = {}
for i in med_text_2: 
    med_text_dict[i] = safe_text_to_disease(i)[1]

In [178]:
#med_text_dict

In [59]:
df.head(3)

,patient_id,preferred_name,gender,date_of_birth,pat_age,pat_created_date,practice_name,provider_name,onboarding_status,rxcui,medication_name,dose,ingredient_name,indication_terms_fda,number_of_refills
0,MTIzNDU2Nzg5MGFiY2RlZg0DoV1+PX+XRLRxIy1FcBS02k...,Joseph Stanley,PATIENT_GENDER_FEMALE,1956-01-14,69,"March 7, 2025, 4:54 PM",Tjalas Healthcare Consulting LLC,Kirk Tjalas,PASSWORD_CREATED,235988.0,venlafaxine hydrochloride,NaN,venlafaxine hydrochloride,"[('anxiety disorder', 3), ('depressive disorde...",1
1,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8On...,Diana Shelton,PATIENT_GENDER_MALE,1994-03-07,31,"January 3, 2025, 1:29 PM",Matthew A Berger MD PC,Matthew Berger,ONBOARDED,1086768.0,vilazodone hydrochloride,10 mg,vilazodone hydrochloride,"[('depressive disorder', 2), ('mdd', 2)]",1
2,MTIzNDU2Nzg5MGFiY2RlZg1TxV54CAnSB1kX/DdDj5K8On...,Diana Shelton,PATIENT_GENDER_MALE,1994-03-07,31,"January 3, 2025, 1:29 PM",Matthew A Berger MD PC,Matthew Berger,ONBOARDED,1099870.0,divalproex sodium 500 mg delayed release oral ...,NaN,divalproex sodium,"[('divalproex sodium extended-release', 5), ('...",9


In [60]:
def fill_indication_from_medication(row):
    value = row['indication_terms_fda']

    # Handle if value is None or an empty list ([])
    if value is None or (isinstance(value, list) and len(value) == 0):
        for key in med_text_dict:
            if key.lower() in str(row['medication_name']).lower():
                return med_text_dict[key]
        return value  # Return original (None or empty list) if no match found
    else:
        return value  # Leave non-null values unchanged

# Apply to the DataFrame
df['indication_terms_fda'] = df.apply(fill_indication_from_medication, axis=1)


In [61]:
df['is_depression_related'] = df['indication_terms_fda'].astype(str).str.contains(r'\b(depressive|MDD)\b', case=False, na=False)

df[df['indication_terms_fda'].isna()].shape
# The indication is null now only for 137 rows, which is 4% of 3333. Hence acceptable. 

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_94154/2254334385.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['is_depression_related'] = df['indication_terms_fda'].astype(str).str.contains(r'\b(depressive|MDD)\b', case=False, na=False)


(116, 16)

In [62]:
#df.to_csv('test_mdd_patient.csv')

In [63]:
df.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'rxcui', 'medication_name', 'dose',
       'ingredient_name', 'indication_terms_fda', 'number_of_refills',
       'is_depression_related'],
      dtype='object')

In [64]:
df_2 = df[['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status','medication_name','dose','is_depression_related','number_of_refills']]

In [65]:
#df_2['Medication_Name_&_Dosage'] = df_2['medication_name']+ ';;; Dose- ' +df_2['dose']
df_2['Medication_Name_&_Dosage'] = df_2['medication_name'].str.cat(df_2['dose'], sep=';;; Dose -', na_rep='*')
df_2['Medication_Name_Dosage_&_Type'] = df_2['Medication_Name_&_Dosage'].str.cat(
    df_2['is_depression_related'].astype(str), sep=';;; Depressive- ', na_rep='*')

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_94154/2372296587.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2['Medication_Name_&_Dosage'] = df_2['medication_name'].str.cat(df_2['dose'], sep=';;; Dose -', na_rep='*')
/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_94154/2372296587.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2['Medication_Name_Dosage_&_Type'] = df_2['Medication_Name_&_Dosage'].str.cat(


In [66]:
#for i in df['pat_created_date']:
df_2['pat_created_date'] = pd.to_datetime(df_2['pat_created_date']).dt.date

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_94154/2688044427.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_2['pat_created_date'] = pd.to_datetime(df_2['pat_created_date']).dt.date
/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_94154/2688044427.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2['pat_created_date'] = pd.to_datetime(df_2['pat_created_date']).dt.date


In [67]:
df_2 = df_2.sort_values(by=['pat_created_date','Medication_Name_Dosage_&_Type'])

In [115]:
df_2.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'medication_name', 'dose', 'is_depression_related',
       'number_of_refills', 'Medication_Name_&_Dosage',
       'Medication_Name_Dosage_&_Type', 'med_flag'],
      dtype='object')

In [124]:
df_2.head(3)

,patient_id,preferred_name,gender,date_of_birth,pat_age,pat_created_date,practice_name,provider_name,onboarding_status,medication_name,dose,is_depression_related,number_of_refills,Medication_Name_&_Dosage,Medication_Name_Dosage_&_Type,med_flag
1216,MTIzNDU2Nzg5MGFiY2RlZlVNZrBxMZ2ktAZSJDpfq3Fqjl...,Caitlin Conley,PATIENT_GENDER_FEMALE,1987-04-04,38,2024-01-25,The Groover Clinic,Ann Groover,ONBOARDED,alprazolam 0.5 mg oral tablet [xanax],1 tbl,False,1,alprazolam 0.5 mg oral tablet [xanax];;; Dose ...,alprazolam 0.5 mg oral tablet [xanax];;; Dose ...,1
1217,MTIzNDU2Nzg5MGFiY2RlZlVNZrBxMZ2ktAZSJDpfq3Fqjl...,Caitlin Conley,PATIENT_GENDER_FEMALE,1987-04-04,38,2024-01-25,The Groover Clinic,Ann Groover,ONBOARDED,alprazolam 0.5 mg oral tablet,1.0 tbl,False,1,alprazolam 0.5 mg oral tablet;;; Dose -1.0 tbl,alprazolam 0.5 mg oral tablet;;; Dose -1.0 tbl...,1
700,MTIzNDU2Nzg5MGFiY2RlZjFDHdkGo80ym5Xvy7V3NfNKwg...,Donna Herrera,PATIENT_GENDER_FEMALE,1987-11-28,37,2024-02-20,The Groover Clinic,MeriDee McArthur,SMS_INVITE_DELIVERY_SENT,amphetamine aspartate,NaN,False,1,amphetamine aspartate;;; Dose -*,amphetamine aspartate;;; Dose -*;;; Depressive...,1


In [90]:
df_2['med_flag'] = 1

# Step 3: Pivot for presence
presence_pivot = df_2.pivot_table(index='patient_id',
                                 columns='Medication_Name_Dosage_&_Type',
                                 values='med_flag',
                                 aggfunc='max',
                                 fill_value=0)

# Step 4: Pivot for number of refills
refill_pivot = df_2.pivot_table(index='patient_id',
                               columns='Medication_Name_Dosage_&_Type',
                               values='number_of_refills',
                               aggfunc='max',  # max or sum depending on logic
                               fill_value=0)

# Step 5: Rename columns to differentiate
presence_pivot.columns = [f"{col}__present" for col in presence_pivot.columns]
refill_pivot.columns = [f"{col}__refills" for col in refill_pivot.columns]

# Step 6: Combine both pivot tables
combined = pd.concat([presence_pivot, refill_pivot], axis=1)

# Optional: sort columns so each medication’s present & refill columns appear together
combined = combined.reindex(sorted(combined.columns), axis=1)

# Step 7: Add patient metadata (optional)
patient_info = df_2.drop_duplicates(subset='patient_id')[[
    'patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
    'pat_created_date', 'practice_name', 'provider_name', 'onboarding_status'
]]

# Final merge
final_df = pd.merge(patient_info, combined, on='patient_id', how='left')

In [91]:
final_df.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status',
       '1 ml diphenhydramine hydrochloride 50 mg/ml prefilled syringe;;; Dose -50 mg;;; Depressive- False__present',
       ...
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__refills',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -*;;; Depressive- False__present',
       'zyprexa;;; Dose -*;;; Depressive- False__refills',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__refills'],
      dtype='object', length=1935)

In [92]:
final_df.shape

(457, 1935)

In [93]:
#final_df.to_csv('Patient_Level_MDD_V1.csv')

In [94]:
df_3 = final_df

In [95]:
list_columns = df_3.columns

In [96]:
print(list_columns)

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status',
       '1 ml diphenhydramine hydrochloride 50 mg/ml prefilled syringe;;; Dose -50 mg;;; Depressive- False__present',
       ...
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__refills',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -*;;; Depressive- False__present',
       'zyprexa;;; Dose -*;;; Depressive- False__refills',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__refills'],
      dtype='object', length=1935)


In [97]:
len(list_columns)

1935

In [98]:
terms = ['dose', 'depressive', 'present']

# Filter strings where ALL terms are present (case-insensitive)
filtered = [text for text in list_columns if all(term.lower() in text.lower() for term in terms)]

print(len(filtered))


963


In [99]:
df_3['total_drugs'] = df_3[filtered].sum(axis=1)

In [100]:
df_3.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status',
       '1 ml diphenhydramine hydrochloride 50 mg/ml prefilled syringe;;; Dose -50 mg;;; Depressive- False__present',
       ...
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__refills',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -*;;; Depressive- False__present',
       'zyprexa;;; Dose -*;;; Depressive- False__refills',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__refills', 'total_drugs'],
      dtype='object', length=1936)

In [101]:
col = 'total_drugs'
cols = list(df_3.columns)
cols.insert(9, cols.pop(cols.index(col)))
df_3 = df_3[cols]

In [102]:
df_3.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'total_drugs',
       ...
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -5 mg;;; Depressive- False__refills',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__present',
       'zolpidem tartrate;;; Dose -6.25 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -*;;; Depressive- False__present',
       'zyprexa;;; Dose -*;;; Depressive- False__refills',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -10 mg;;; Depressive- False__refills',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__present',
       'zyprexa;;; Dose -5 mg;;; Depressive- False__refills'],
      dtype='object', length=1936)

In [107]:
df_3.sort_values(by='total_drugs',inplace=True)

In [109]:
#df_3.to_csv('Patient_Level_MDD_V3.csv')

In [155]:
df_2.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'medication_name', 'dose', 'is_depression_related',
       'number_of_refills', 'Medication_Name_&_Dosage',
       'Medication_Name_Dosage_&_Type', 'med_flag'],
      dtype='object')

### Change Log 1: On June 25, Erwin asked to only pull drugs that are related to Depression. So that ask is done in Path 1 logic, the other
### is in Path 2 logic. If Path 1 (removal of depression) is not required then use Path 2. 

In [203]:
list_features = ['patient_id','preferred_name','gender','date_of_birth','pat_age','pat_created_date','practice_name',
                 'provider_name','onboarding_status','Medication_Name_Dosage_&_Type','number_of_refills']
# Path 1
df_3 = df_2.loc[df_2['is_depression_related'] == True,list_features]
    # The shape of this path is - (1525, 11)
    # The patients count - 407

#Path 2
#df_3 = df_2[['patient_id','preferred_name','gender','date_of_birth','pat_age',
#            'pat_created_date','practice_name','provider_name','onboarding_status',
#            'Medication_Name_Dosage_&_Type','number_of_refills']]
    # The shape of this path - (3333,11)
    # The patient count - 457

In [204]:
len(df_3['patient_id'].unique())

407

In [205]:
#import pandas as pd  

'''
# Sample data creation for illustration  
data = {  
    'patient_id': [1, 1, 2, 2, 3],  
    'preferred_name': ['John', 'John', 'Alice', 'Alice', 'Bob'],  
    'gender': ['M', 'M', 'F', 'F', 'M'],  
    'date_of_birth': ['1990-01-01', '1990-01-01', '1985-05-15', '1985-05-15', '2000-12-12'],  
    'pat_age': [31, 31, 36, 36, 21],  
    'pat_created_date': ['2020-01-01', '2020-01-01', '2019-06-01', '2019-06-01', '2021-07-07'],  
    'practice_name': ['Practice A', 'Practice A', 'Practice B', 'Practice B', 'Practice C'],  
    'provider_name': ['Dr. Smith', 'Dr. Smith', 'Dr. Jones', 'Dr. Jones', 'Dr. Taylor'],  
    'onboarding_status': ['Completed', 'Completed', 'Pending', 'Pending', 'Completed'],  
    'Medication_Name_Dosage_&_Type': ['Med1', 'Med2', 'Med3', 'Med4', 'Med5'],  
    'number_of_refills': [3, 1, 5, 2, 4]  
}  
  
# Create dataframe  
df = pd.DataFrame(data)
'''
# Step 1: Sort by patient + refills descending
df_3_sorted = df_3.sort_values(by=['patient_id', 'number_of_refills'], ascending=[True, False])

# Step 2: Enumerate drug ranks per patient
df_3_sorted['drug_rank'] = df_3_sorted.groupby('patient_id').cumcount() + 1

# Step 3: Pivot to wide format
pivoted = df_3_sorted.pivot(index='patient_id', 
                          columns='drug_rank', 
                          values=['Medication_Name_Dosage_&_Type', 'number_of_refills'])

# Step 4: Flatten multi-index columns and interleave Drug + Refill
drugs = pivoted['Medication_Name_Dosage_&_Type']
refills = pivoted['number_of_refills']

# Reconstruct interleaved column order
interleaved_cols = []
for i in sorted(drugs.columns):
    interleaved_cols.append(('Medication_Name_Dosage_&_Type', i))
    interleaved_cols.append(('number_of_refills', i))

# Reorder pivoted columns
pivoted = pivoted[interleaved_cols]

# Rename columns to 'Drug {i}' and 'Drug {i} Refills'
pivoted.columns = [f"Drug {i//2 + 1}" if j % 2 == 0 else f"Drug {i//2 + 1} Refills" 
                   for i, j in enumerate(range(len(pivoted.columns)))]

# Add Patient ID back
pivoted.reset_index(inplace=True)

pivoted.reset_index(inplace=True)

# Step 5: Merge with static patient data
patient_info_cols = [
    'patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
    'pat_created_date', 'practice_name', 'provider_name', 'onboarding_status'
]
patient_info_df = df_3[patient_info_cols].drop_duplicates(subset='patient_id')

final_df = pd.merge(patient_info_df, pivoted, on='patient_id', how='left')

# Count number of non-null drugs per patient
drug_cols = [col for col in final_df.columns if col.startswith("Drug ") and "Refills" not in col]
final_df["total_drugs"] = final_df[drug_cols].notna().sum(axis=1)

# Sort by number of drugs (descending)
final_df = final_df.sort_values(by="total_drugs", ascending=True).reset_index(drop=True)


In [206]:
final_df.columns

Index(['patient_id', 'preferred_name', 'gender', 'date_of_birth', 'pat_age',
       'pat_created_date', 'practice_name', 'provider_name',
       'onboarding_status', 'index', 'Drug 1', 'Drug 1 Refills', 'Drug 2',
       'Drug 2 Refills', 'Drug 3', 'Drug 3 Refills', 'Drug 4',
       'Drug 4 Refills', 'Drug 5', 'Drug 5 Refills', 'Drug 6',
       'Drug 6 Refills', 'Drug 7', 'Drug 7 Refills', 'Drug 8',
       'Drug 8 Refills', 'Drug 9', 'Drug 9 Refills', 'Drug 10',
       'Drug 10 Refills', 'Drug 11', 'Drug 11 Refills', 'Drug 12',
       'Drug 12 Refills', 'Drug 13', 'Drug 13 Refills', 'Drug 14',
       'Drug 14 Refills', 'Drug 15', 'Drug 15 Refills', 'Drug 16',
       'Drug 16 Refills', 'Drug 17', 'Drug 17 Refills', 'Drug 18',
       'Drug 18 Refills', 'Drug 19', 'Drug 19 Refills', 'Drug 20',
       'Drug 20 Refills', 'Drug 21', 'Drug 21 Refills', 'Drug 22',
       'Drug 22 Refills', 'Drug 23', 'Drug 23 Refills', 'total_drugs'],
      dtype='object')

In [207]:
df_2[df_2['patient_id'] == 'MTIzNDU2Nzg5MGFiY2RlZnwsg6vkdtdByEewlUR+uFuaW1X0lr8oOR/VDtgj0NydbxsauajItWzN16IOl3B+TuUc76VjtdYXkvbN6azlhAXAsTgRbzJd/89fOkaLI9n1']

,patient_id,preferred_name,gender,date_of_birth,pat_age,pat_created_date,practice_name,provider_name,onboarding_status,medication_name,dose,is_depression_related,number_of_refills,Medication_Name_&_Dosage,Medication_Name_Dosage_&_Type,med_flag
1643,MTIzNDU2Nzg5MGFiY2RlZnwsg6vkdtdByEewlUR+uFuaW1...,Michael Douglas,PATIENT_GENDER_FEMALE,1972-06-25,52,2025-05-22,Kokoro Health,Shannon Eaton,PASSWORD_CREATED,24 hr desvenlafaxine succinate 50 mg extended ...,1 tbl,True,1,24 hr desvenlafaxine succinate 50 mg extended ...,24 hr desvenlafaxine succinate 50 mg extended ...,1
1642,MTIzNDU2Nzg5MGFiY2RlZnwsg6vkdtdByEewlUR+uFuaW1...,Michael Douglas,PATIENT_GENDER_FEMALE,1972-06-25,52,2025-05-22,Kokoro Health,Shannon Eaton,PASSWORD_CREATED,hydroxyzine hydrochloride,NaN,False,1,hydroxyzine hydrochloride;;; Dose -*,hydroxyzine hydrochloride;;; Dose -*;;; Depres...,1
1645,MTIzNDU2Nzg5MGFiY2RlZnwsg6vkdtdByEewlUR+uFuaW1...,Michael Douglas,PATIENT_GENDER_FEMALE,1972-06-25,52,2025-05-22,Kokoro Health,Shannon Eaton,PASSWORD_CREATED,trazodone hydrochloride,50 mg,True,1,trazodone hydrochloride;;; Dose -50 mg,trazodone hydrochloride;;; Dose -50 mg;;; Depr...,1
1644,MTIzNDU2Nzg5MGFiY2RlZnwsg6vkdtdByEewlUR+uFuaW1...,Michael Douglas,PATIENT_GENDER_FEMALE,1972-06-25,52,2025-05-22,Kokoro Health,Shannon Eaton,PASSWORD_CREATED,zolpidem tartrate,NaN,False,1,zolpidem tartrate;;; Dose -*,zolpidem tartrate;;; Dose -*;;; Depressive- False,1


In [208]:
if 'index' in final_df.columns:
    final_df.drop(columns='index', inplace=True)

# Step 2: Move 'total_drugs' to the position where 'index' was (before 'Drug 1')
# Find current index of 'Drug 1'
drug1_pos = final_df.columns.get_loc('Drug 1') if 'Drug 1' in final_df.columns else len(final_df.columns)

# Reorder columns
cols = final_df.columns.tolist()
cols.insert(drug1_pos, cols.pop(cols.index('total_drugs')))
final_df = final_df[cols]

In [209]:
#final_df.to_csv('patient_level_MDD_V4.csv')

In [210]:
final_df

,patient_id,preferred_name,gender,date_of_birth,pat_age,pat_created_date,practice_name,provider_name,onboarding_status,total_drugs,...,Drug 19,Drug 19 Refills,Drug 20,Drug 20 Refills,Drug 21,Drug 21 Refills,Drug 22,Drug 22 Refills,Drug 23,Drug 23 Refills
0,MTIzNDU2Nzg5MGFiY2RlZphBkdIgb91FYxvnT9A4X6RLoO...,Bradley Collins,PATIENT_GENDER_FEMALE,1987-08-06,37,2025-01-14,Matthew A Berger MD PC,Abigail Skene,ONBOARDED,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MTIzNDU2Nzg5MGFiY2RlZgricfouO5rB064qMxk3xZbiA4...,Helen Campos,PATIENT_GENDER_FEMALE,1986-07-31,38,2025-01-11,Visionary Psychiatry,Crystal Higbe,PASSWORD_CREATED,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MTIzNDU2Nzg5MGFiY2RlZsjb7QAtDnBHuOwgTxyR2P0eI3...,Margaret Lane,PATIENT_GENDER_MALE,1991-04-18,34,2025-04-09,Visionary Psychiatry,Carmen Kosicek,PASSWORD_CREATED,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,MTIzNDU2Nzg5MGFiY2RlZuHANkrzF4zm3ctx+7A0vhv5z8...,Michael Monroe,PATIENT_GENDER_FEMALE,1977-01-21,48,2025-03-07,Tjalas Healthcare Consulting LLC,Kirk Tjalas,PASSWORD_CREATED,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MTIzNDU2Nzg5MGFiY2RlZjD+3AbUrYv/8FnoaQIsXm+eGN...,Patrick Gross,PATIENT_GENDER_FEMALE,1958-02-12,67,2025-01-15,Matthew A Berger MD PC,Matthew Berger,ONBOARDED,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
402,MTIzNDU2Nzg5MGFiY2RlZpB1PPq1vmK94b2B86dLuA1XrG...,Jack Patel,PATIENT_GENDER_FEMALE,2004-04-18,21,2025-02-04,Authentic Psychiatric Services,Jonathan Hoerl,PASSWORD_CREATED,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
403,MTIzNDU2Nzg5MGFiY2RlZgI4UmtDkkVSo00TP3r8kCn+9i...,Nicole Myers MD,PATIENT_GENDER_MALE,1965-06-24,59,2025-01-31,Visionary Psychiatry,Catherine Hensy,PASSWORD_CREATED,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
404,MTIzNDU2Nzg5MGFiY2RlZuQPYV9v+yRL5slKOARJKJqybq...,John Nash,PATIENT_GENDER_FEMALE,1976-02-07,49,2024-10-16,Matthew A Berger MD PC,Matthew Berger,ONBOARDED,16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
405,MTIzNDU2Nzg5MGFiY2RlZu50m1vEcVPfCyh2lfYCg8j145...,Mrs. Ruth Russell,PATIENT_GENDER_FEMALE,1980-03-15,45,2024-10-21,Matthew A Berger MD PC,Melanie Priebe,ONBOARDED,22,...,venlafaxine hcl er 150 mg cap;;; Dose -*;;; De...,1,venlafaxine hcl er 37.5 mg cap;;; Dose -*;;; D...,1,venlafaxine hcl er 75 mg cap;;; Dose -*;;; Dep...,1,venlafaxine hydrochloride;;; Dose -300 mg;;; D...,1,NaN,NaN


In [211]:
#final_df.to_csv('patient_level_MDD_V5.csv')